# 01 — Statistical foundation

Observed agreement \(A_o\) vs expected agreement under independence \(A_e = \sum_k \pi_k^2\).

- Theory: `notes/phase1-theory.md`  
- Generator: `src/simulate.py`  
- Metrics: `src/metrics.py`  

Run this notebook from the **repository root** (or ensure the project root is on `PYTHONPATH`).

In [ ]:
from pathlib import Path
import os, sys

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("Project root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd

from src.simulate import SimulatedAnnotation
from src.metrics import observed_agreement, expected_agreement_independence, observed_agreement_global_with_replacement

## Toy matrix: \(A_o = 2/3\)

Same 3-item, 2-annotator example as in `notes/phase1-theory.md`.

In [ ]:
df_toy = pd.DataFrame([[0, 0], [1, 0], [2, 2]])
observed_agreement(df_toy), 2 / 3

## Pure random labellers

Each cell i.i.d. from \(\pi\) (`pure_random=True`). For large \(n\), empirical \(A_o\) should approach \(A_e = \sum_k \pi_k^2\).

In [ ]:
gen = SimulatedAnnotation(
    n_items=10_000,
    n_annotators=5,
    n_classes=3,
    class_dist="uniform",
    pure_random=True,
    seed=42,
)
df = gen.generate()
ao = observed_agreement(df)
ae = expected_agreement_independence([1 / 3, 1 / 3, 1 / 3])
ao, ae, abs(ao - ae)

In [ ]:
pi = np.array([0.70, 0.20, 0.10])
gen_i = SimulatedAnnotation(
    n_items=10_000,
    n_annotators=5,
    n_classes=3,
    class_dist=pi,
    pure_random=True,
    seed=7,
)
df_i = gen_i.generate()
ao_i = observed_agreement(df_i)
ae_i = expected_agreement_independence(pi)
ao_i, ae_i, float(ae_i)

## Global with-replacement view

Compare `observed_agreement` (within-item pairs) to pooling all judgments.

In [ ]:
observed_agreement(df), observed_agreement_global_with_replacement(df)

## Convergence figure

Regenerates `figures/random_agreement_convergence.png` (same logic as `scripts/phase1_convergence_plot.py`).

In [ ]:
import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

n_grid = [100, 500, 1000, 5000, 10_000]
n_annotators, n_classes = 5, 3
pi_imb = np.array([0.70, 0.20, 0.10])
ae_u = expected_agreement_independence(np.full(n_classes, 1.0 / n_classes))
ae_i = expected_agreement_independence(pi_imb)

aos_u, aos_i = [], []
for idx, n_items in enumerate(n_grid):
    g_u = SimulatedAnnotation(
        n_items, n_annotators, n_classes, class_dist="uniform", pure_random=True, seed=42 + idx
    )
    aos_u.append(observed_agreement(g_u.generate()))
    g_i = SimulatedAnnotation(
        n_items, n_annotators, n_classes, class_dist=pi_imb, pure_random=True, seed=100 + idx
    )
    aos_i.append(observed_agreement(g_i.generate()))

fig, ax = plt.subplots(figsize=(8, 5), dpi=120)
ax.plot(n_grid, aos_u, "o-", label="Empirical $A_o$ (uniform i.i.d.)")
ax.axhline(ae_u, linestyle="--", label=f"$A_e = 1/K = {ae_u:.4f}$")
ax.plot(n_grid, aos_i, "s-", label=r"Empirical $A_o$ (i.i.d. $\pi=(0.7,0.2,0.1)$)")
ax.axhline(ae_i, linestyle="--", label=f"$A_e = \\sum \\pi_k^2 = {ae_i:.4f}$")
ax.set_xscale("log")
ax.set_xticks(n_grid)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel("Number of items $n$")
ax.set_ylabel(r"Observed agreement $A_o$")
ax.set_title("Convergence of empirical $A_o$ under independent random labelling")
ax.legend()
fig.tight_layout()

out = ROOT / "figures" / "random_agreement_convergence.png"
fig.savefig(out, bbox_inches="tight")
plt.show()
print("Saved", out)